# Stage Two - The Fine-Tuned Bi-Encoder

**What stage one gave you.** A measuring instrument you have verified against a
published number. Every result from here is trustworthy because of that.

**What stage two does.** It attacks **failure mode A** - documents that never made
the shortlist at all. Fine-tuning *moves* documents in the embedding space, which
can pull things into the top 100 that were not there before. This is why
Recall@100 can improve here, and why it mathematically *cannot* improve from
reranking in stage three.

**The experiment inside it.** You run the whole pipeline twice on the same
dataset:

| Arm | Trained on | Represents |
|---|---|---|
| **A** `real_labels` | the human-labelled pairs BEIR ships | the ceiling: what a perfect answer key buys you |
| **B** `synthetic` | generated queries only, pretending labels do not exist | what you could actually do at a company |

**The gap between them is the headline finding.** It answers the question anyone
in this situation actually has: *how much does not having labels cost me?* That
is a real answer to a real question, and it holds regardless of whether you hit
any particular absolute number.

**The method has a name.** GPL - *Generative Pseudo Labeling for Unsupervised
Domain Adaptation of Dense Retrieval*, Wang et al., 2022. Read it and cite it.
It signals you know the technique has a literature.

---

### The four steps, and where each runs

| Step | File | Needs a GPU? | Roughly |
|---|---|---|---|
| 2. Generate queries | `src/generate_queries.py` | **yes** | 20-45 min |
| 3. Mine negatives | `src/mine_negatives.py` | helps | 2 min |
| 4. Fine-tune | `src/train.py` | **yes** | 5-20 min |
| re-index + evaluate | stage one's files, unchanged | helps | minutes |

Measured on this machine: one training step on CPU took **50 seconds**. Do
steps 2 and 4 on Colab.

## 0. Setup

Same as stage one: find the project root, put `src/` on the path.

In [ ]:
import os
import sys
from pathlib import Path

here = Path.cwd()
while not (here / "configs").exists() and here != here.parent:
    here = here.parent
os.chdir(here)
sys.path.insert(0, str(here / "src"))

import json
import numpy as np
import torch

from config import load_config
from ingest import doc_text, load_raw_corpus

print("project root:", Path.cwd())
print("GPU available:", torch.cuda.is_available())

Stage two is per-dataset, and the dataset choice is not arbitrary.

**NFCorpus is the headline.** Its queries are health questions written by
non-specialists; its documents are PubMed abstracts in clinical vocabulary. The
two sides describe the same things in different words, and closing that distance
is exactly what fine-tuning does.

**SciFact will probably gain little or nothing, and that is predicted.** Its
queries are scientific claims written in the same register as the abstracts, so
there is barely a gap to close. Your own stage one numbers say so: BM25 scored
0.65 there, and BM25 can only score what two texts literally share. High BM25
means the vocabularies already coincide.

**Never train on FiQA.** It is the control. More on that in section 6.

In [ ]:
# Point this at nfcorpus once you have frozen its split. We use scifact below
# only because it is the dataset already ingested on this machine.
cfg = load_config("configs/scifact.yaml")
print(cfg.summary())

## 1. Step 2 - Manufacturing training queries

**The premise.** You have documents and no training questions. At a company that
is the normal situation: a corpus exists, nobody ever wrote down which query
should retrieve which document.

**The move.** Show a generator a passage and ask what someone might have typed to
find it. Three to five per document.

Two model choices:

- `BeIR/query-gen-msmarco-t5-base-v1` - a T5 trained by the BEIR authors for
  exactly this. Reliable, fast, and the default in `generate_queries.py`.
- A small instruction model such as `Qwen2.5-1.5B-Instruct`, prompted for
  different *styles* from one passage: one keyword query, one full question, one
  vague underspecified one. Better matches how real people type, and gives you
  something to discuss in the write-up. An upgrade, not a prerequisite.

### Design choice: sampling, not beam search

Beam search returns the *n most probable* sequences. For one passage those are
five near-identical rephrasings of the same question, which teaches the model
nothing beyond what one query would.

Sampling with `top_p=0.95` and `temperature=1.0` gives genuinely different
questions about different aspects of the passage. That is the entire reason to
generate several rather than one.

In [ ]:
# The generation call, from src/generate_queries.py. Not executed here - it
# wants a GPU and a ~900MB model download. Read it, then run it on Colab.
print('''
out = generator.generate(
    **encoded_passages,
    max_length=64,
    do_sample=True,        # <- NOT beam search
    top_p=0.95,            #    nucleus sampling: keep the smallest set of
    temperature=1.0,       #    tokens whose probability sums to 0.95
    num_return_sequences=n_per_doc,
)
''')

### Design choice: the round-trip filter, which is not optional

A generator hallucinates. It writes questions the passage does not answer,
questions about a detail that appears in a thousand other passages, and questions
that are simply incoherent. Training on those teaches the model to associate a
query with a document that does not answer it, which is **worse than not training
at all**.

The test is cheap and direct:

> Embed the generated query with the **base** model, search the corpus, and ask
> whether the passage it came from comes back in the top 10. If not, throw the
> pair away.

It typically discards 15-30% of generations and is the cheapest quality
improvement in the whole pipeline.

Note it uses the **base** model deliberately. Filtering with the model you are
about to train on this data would be circular.

In [ ]:
# The filter, in miniature, using the base model and the stage-one index.
import faiss
from sentence_transformers import SentenceTransformer

stem = Path("data/embeddings/scifact_base")
index = faiss.read_index(str(stem.with_suffix(".faiss")))
doc_ids = json.loads(stem.with_suffix(".ids.json").read_text())
position_of = {d: i for i, d in enumerate(doc_ids)}

corpus = load_raw_corpus(cfg.dataset)
encoder = SentenceTransformer(cfg.base_model)
encoder.max_seq_length = cfg.max_seq_length

source_id = sorted(corpus)[0]
print("source passage:", corpus[source_id]["title"][:70], "\n")

# Pretend these three came out of the generator for that passage:
candidates = [
    # on-topic and specific - a plausible good generation
    corpus[source_id]["title"],
    # on-topic but so generic it describes a thousand other papers
    "what does this study measure",
    # a hallucination: nothing to do with the passage
    "what is the capital city of France",
]

vecs = encoder.encode([cfg.for_query(q) for q in candidates],
                      normalize_embeddings=True).astype(np.float32)
_, positions = index.search(vecs, 10)

for q, row in zip(candidates, positions):
    found = position_of[source_id] in row
    print(f"{'KEEP   ' if found else 'DISCARD'}  {q[:72]}")

The first survives because the base model can already find its source passage
from it, so the pair is learnable signal. The other two are discarded - one
because it describes a thousand papers equally well, one because it is a
hallucination.

That is the whole test. Cheap, mechanical, and it removes the generations that
would otherwise teach the model to associate a query with a document that does
not answer it.

### The failure mode to watch for

The generator copies passage vocabulary verbatim. The training task then collapses
into lexical matching, and the model learns nothing about *meaning* - it just
learns to match words it can already match.

**The symptom during training:** loss drops to near zero within a few hundred
steps. If you see that, stop and look at your generated queries.

**The prevention:** read some by hand. Always. `generate_queries.py` prints a
reminder to do this, because it is the step everyone skips.

### Cost, and the rule that follows from it

This is the slowest step in the project. About 15,000 generations takes 20-45
minutes on a free T4 with T5-base.

So: **generate once, save to disk, never regenerate.** `generate_queries.py`
refuses to overwrite an existing file without `--force`, because its output is
what every later result was trained on.

In [ ]:
print("Run this on Colab, once:")
print()
print("  python src/generate_queries.py --config configs/nfcorpus.yaml \\")
print("      --n-per-doc 3")
print()
print("Writes data/synthetic_queries/nfcorpus.jsonl plus a .stats.json")
print("recording the discard rate. Check that rate - if it is near zero your")
print("filter is not working, and if it is near 100% your generator is.")

## 2. Step 3 - Hard negative mining

Contrastive training needs three things: a query, its correct document, and
several **wrong** ones.

**Random wrong documents are useless.** Pair *"does eating eggs raise
cholesterol"* against a paper on soil pH and the model separates them trivially.
It can already tell them apart, so the gradient is near zero and it learns
nothing.

What teaches the model something is a document that is *plausibly* relevant and
is not. Those live just below the top of the ranking.

### Design choice 1: take ranks 10 to 50, skipping the top

**Why skip the top 10?** Your answer key is incomplete. BEIR labels a fraction of
the genuinely relevant documents, and the unlabelled ones concentrate *right at
the top* of a good ranking - that is what makes the ranking good.

Scoop those up as "negatives" and you actively train the model that correct
answers are wrong. That is not merely wasted effort, it is harmful.

**Why stop at 50?** Below that the documents stop being plausible and start being
obviously wrong, which brings you back to the random-negatives problem.

### Design choice 2: the margin filter

A "negative" scoring nearly as high as the positive probably *is* relevant and
simply was not labelled. So require every negative to score at least 5% below the
positive. Cheap, and it catches what the positional rule misses.

Both numbers live in the config, under `mining:`.

In [ ]:
print(json.dumps({
    "range_min": cfg.mining.range_min,
    "range_max": cfg.mining.range_max,
    "margin": cfg.mining.margin,
    "negatives_per_query": cfg.negatives_per_query,
}, indent=2))

### What it produced on real data

`src/mine_negatives.py` has already been run on SciFact's real-label arm. Here is
what one training example looks like.

In [ ]:
neg_path = Path("data/hard_negatives/scifact_real_labels_base.jsonl")
rows = [json.loads(l) for l in neg_path.read_text(encoding="utf-8").splitlines()]
print(f"{len(rows)} training rows\n")

r = rows[0]
print("QUERY    ", r["query"][:90])
print()
print("POSITIVE ", corpus[r["positive"]]["title"][:80])
print()
print("NEGATIVES (plausible, but wrong):")
for d in r["negatives"]:
    print("  -", corpus[d]["title"][:76])

Read those negatives. They are all biology papers, all superficially on-topic,
and none of them answer the query. That is what "hard" means. A random negative
would be a paper on economics, and the model would separate it without learning
anything.

### The filter statistics are worth checking

In [ ]:
stats_seen = {
    "pairs_in": 200,
    "pairs_out": len(rows),
    "candidates_dropped_by_margin": 1932,
    "dropped_no_negatives": 200 - len(rows),
}
print(json.dumps(stats_seen, indent=2))
print()
print("1932 candidates were dropped for scoring too close to the positive.")
print("On SciFact that number is high, and it makes sense: most queries have")
print("exactly one labelled relevant document, so plenty of genuinely relevant")
print("papers sit unlabelled in ranks 10-50. The margin filter is catching them.")

### Optional upgrade

Score the candidates with an off-the-shelf cross-encoder such as
`cross-encoder/ms-marco-MiniLM-L-6-v2` and drop anything it rates clearly
relevant. That is *real* false-negative filtering rather than a positional
heuristic, and at this scale it takes minutes.

### The failure mode: not re-mining

After fine-tuning, the model has learned to push these specific negatives down.
They are not hard any more.

Re-mine with the **tuned** model's index and train a second round. It usually
gains again, and reporting both rounds is a stronger result than reporting one.

```
python src/mine_negatives.py --config configs/nfcorpus.yaml \
    --arm synthetic --index-tag synthetic_s42
```

## 3. Step 4 - The contrastive fine-tune

The signal is simple: **pull the query toward its correct document, push it away
from the negatives.** Repeat a few thousand times and the embedding space
reshapes, so that lay phrasing lands near the clinical phrasing that answers it.

The loss is `MultipleNegativesRankingLoss`. Intuitively it treats each batch as a
multiple-choice exam: here is a query and N documents, exactly one is correct,
maximise the probability assigned to the right one.

### The batch size thing, which is not intuitive

In this loss, **every other item in the batch acts as an additional negative.**

So batch size *is* your negative count. That makes it the single most important
hyperparameter, and it is exactly what a 16GB T4 constrains.

`CachedMultipleNegativesRankingLoss` (GradCache) works around it: compute
embeddings in mini-batches, cache them, then reconstruct the gradient of a large
batch. An effective batch of 256 with the memory footprint of 16.

**Running 64 versus 256 and reporting the nDCG difference is a genuinely good
result for one extra run.** `train.py --cached-loss` switches it on.

In [ ]:
print(json.dumps({
    "base_model": cfg.base_model,
    "lr": cfg.training.lr,
    "batch_size": cfg.training.batch_size,
    "epochs": cfg.training.epochs,
    "warmup_ratio": cfg.training.warmup_ratio,
    "max_seq_length": cfg.max_seq_length,
}, indent=2))

### How the training data is laid out

`MultipleNegativesRankingLoss` reads columns **positionally**: the first is the
anchor, the second is its positive, every remaining column is a negative. Column
names do not matter; order does.

And this is where the prefixes are applied - through `cfg.for_query` and
`cfg.for_document`, the same two functions retrieval uses. Train with a different
prefix than you evaluate with and the model is optimised for inputs it will never
see. Nothing would report that.

In [ ]:
from train import build_dataset

ds = build_dataset(cfg, corpus, neg_path, max_rows=4)
print("columns:", ds.column_names)
print()
row = ds[0]
print("anchor    :", row["anchor"][:95])
print("positive  :", row["positive"][:95])
print("negative_1:", row["negative_1"][:95])

Notice the anchor carries the BGE prefix and the documents do not. That asymmetry
is the model's contract, and it is applied in exactly one place in the codebase.

### Early stopping on dev, never test

`train.py` builds an `InformationRetrievalEvaluator` over the **dev** split and
scores nDCG@10 every few hundred steps, keeping the best checkpoint.

Dev, never test. If you pick the checkpoint by watching the test score, you
selected on the number you are about to report, and it stops being an estimate of
unseen performance.

### Running it

In [ ]:
print("On Colab, for each arm:")
print()
for arm in ("real_labels", "synthetic"):
    print(f"  python src/train.py --config configs/nfcorpus.yaml --arm {arm}")
print()
print("Measured on this CPU: ~50 seconds per training step. On a T4, roughly")
print("10,000 pairs for 2 epochs at batch 64 takes 5-20 minutes - ten runs in")
print("an afternoon.")

## 4. Re-index and re-evaluate

**A new model means a new embedding space.** Every document vector from the base
model is meaningless to the tuned model, so the whole corpus must be re-encoded.
`retrieve.py` enforces this: it refuses to run if the index was built with a
different model than the one encoding the queries, because mixing them puts
queries and documents in unrelated spaces and the scores still *look* like scores.

Nothing else changes. Same `build_index.py`, same `retrieve.py`, same
`evaluate.py`, same frozen split. That is the entire payoff of stage one - the
harness does not know or care that a model was fine-tuned.

In [ ]:
print('''
python src/build_index.py --config configs/nfcorpus.yaml \\
    --model models/nfcorpus_synthetic_s42 --tag synthetic_s42

python src/retrieve.py --config configs/nfcorpus.yaml --tag synthetic_s42

python src/evaluate.py --config configs/nfcorpus.yaml \\
    --run data/runs/nfcorpus_synthetic_s42_test.trec \\
    --model-name models/nfcorpus_synthetic_s42 \\
    --trained-on nfcorpus --training-arm synthetic \\
    --index-type flat --query-prefix-used yes
''')

Note `--trained-on nfcorpus --training-arm synthetic`. Those two columns are why
`metrics.csv` stays readable. `dataset` is what you *evaluate* on; `trained_on` is
what the model was *tuned* on. Keep them separate and a FiQA row is unambiguous.
Merge them and you cannot tell whether a model was tuned on SciFact or NFCorpus.

## 5. The forgetting check

Fine-tuning a model to be good at nutrition can make it worse at everything else.
It specialises. **If you do not check for this, a reader will ask and you will
have no answer.**

FiQA is the probe because financial text is maximally distant from biology and
nutrition, so any drop is a clean signal rather than a domain-overlap artefact.
At ~58k documents it is also the only set large enough to make an ANN sweep
honest.

### The rule: it is a delta, never an absolute

`0.38` on its own means nothing. `0.40 base -> 0.38 tuned` means fine-tuning cost
you two points elsewhere.

So measure the base model on FiQA **first**, and write the number down.
`scripts/run_forgetting.sh` checks `metrics.csv` for that reference row and
measures it automatically if it is missing.

### Why FiQA is marked eval-only in code, not just in prose

Training on FiQA would destroy the measurement FiQA exists to provide. So
`configs/fiqa.yaml` sets `trainable: false`, and `cfg.require_training()` raises
by name. `train.py` calls that instead of reading `cfg.training` directly, which
makes the mistake structurally impossible.

In [ ]:
from config import ConfigError

fiqa = load_config("configs/fiqa.yaml")
try:
    fiqa.require_training()
except ConfigError as e:
    print("refused:", e)

### Terminology, because a reviewer will notice

This is catastrophic **forgetting**, not "cascade forgetting" - the cascade is the
BM25 to bi-encoder to reranker chain, which is a stage-three concern.

And strictly, what FiQA measures is **out-of-domain generalisation loss**. Call it
what it is.

## 6. Reading the results

### Is forgetting bad?

Some loss is expected and fine. The model has 33M parameters and fixed capacity -
teaching it that lay nutrition phrasing maps to clinical phrasing necessarily
reshapes the space. **A model that loses nothing anywhere probably learned
nothing.**

What matters is the trade, framed as a ratio:

| NFCorpus gain | FiQA loss | Verdict |
|---|---|---|
| +6 | -2 | Good trade, easy to defend |
| +6 | -1 | Excellent |
| +2 | -6 | Bad - you damaged more than you built |
| +6 | -15 | Over-specialised. Lower the LR or the epoch count. |

There is also a context question. If the deployment is a nutrition search system,
forgetting finance is irrelevant - nobody will ask it a finance question.
Out-of-domain loss matters only if you are claiming a general-purpose model.
**State which claim you are making.**

### Is there an optimal base-to-tuned difference?

No. Any threshold would be invented. What exists instead are diagnostic patterns:

| Pattern | What it means |
|---|---|
| Gain ~0 on both | Domain gap too small, or training failed. Check the loss curve. |
| Large gain on SciFact, small on NFCorpus | Backwards from expectation. Likely a leak or a bug. |
| Gain on NFCorpus, ~0 on SciFact | Exactly as predicted. Good. |
| Huge gain, huge forgetting | Over-training. Reduce LR or epochs. |
| **Arm A >> Arm B** | Query generation is your bottleneck. |
| **Arm A ~ Arm B** | Synthetic queries nearly match human labels. Strong finding. |

Those last two rows are the point of the whole project.

### The noise caution

**A gain under about one nDCG point is inside run-to-run variance.** To claim a
small gain, train with three seeds and report mean and spread. Otherwise you may
be reporting a seed rather than a result.

`train.py --seed` exists for exactly this, and `set_all_seeds` pins Python, NumPy
and torch so that a seed actually determines the run.

In [ ]:
print("Three seeds, one arm:")
for s in (42, 43, 44):
    print(f"  python src/train.py --config configs/nfcorpus.yaml "
          f"--arm synthetic --seed {s}")
print()
print("Then compare the three ndcg@10 values in results/metrics.csv.")
print("If their spread is larger than your claimed gain, you have no result yet.")

### The risk worth stating plainly

`bge-small-en-v1.5` was trained on a large, diverse corpus that already includes
scientific text. Fine-tuning on synthetic SciFact queries may produce one or two
nDCG points, or none. **That is a real outcome, not a mistake.**

Three mitigations:

1. Prefer NFCorpus for the headline.
2. Include the real-labels arm. If synthetic training fails but real-label
   training succeeds, *that itself is the finding*.
3. If you report a large gain over a deliberately weak baseline, say so, and
   report the strong baseline alongside.

---

## 7. The whole stage in one command

```
bash scripts/run_stage2.sh configs/nfcorpus.yaml
```

It runs generation, mines both arms, trains both arms, re-indexes, re-evaluates,
and runs the forgetting check for each. Prerequisite: stage one must have run on
that dataset, because you need the frozen split, the base index, and the base row
to compare against.

**Stage three - the cross-encoder reranker - comes after all of this, or not at
all.** It reads a run file and writes a run file, never touching stage two's
code. And remember it attacks failure mode B only: Recall@100 is mathematically
unchanged by reranking, because it is the same 100 documents in a different
order.